In [13]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from pyspark.conf import SparkConf
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col
from urllib import request
import os

In [14]:
pyspark.__file__

'/home/daniel/de_zoomcamp_2026_project/.venv/lib/python3.12/site-packages/pyspark/__init__.py'

In [15]:
#configuration
gcs_credentials = '/home/daniel/de_zoomcamp_2026_project/gcs_credentials/credentials/service_account_creds.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('project_pipeline') \
    .set("spark.driver.memory", "4g") \
    .set("spark.executor.memory", "8g") \
    .set("spark.jars", ",".join([
        "/home/daniel/de_zoomcamp_2026_project/gcs_hadoop_conn/gcs-connector-hadoop3-2.2.5.jar",
        "/home/daniel/de_zoomcamp_2026_project/gcs_hadoop_conn/spark-bigquery-with-dependencies_2.13-0.44.0.jar"
    ])) \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", gcs_credentials)

In [16]:
#context
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", gcs_credentials)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/26 04:37:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [17]:
#Creating Spark session
spark = SparkSession.builder \
        .config(conf= sc.getConf()) \
        .getOrCreate()

In [18]:
bucket = "de-zoomcamp-2026-project-bucket"
spark.conf.set("temporaryGcsBucket", bucket)

In [19]:
df_housing_gcs = \
    spark.read \
    .option('header', 'true') \
    .parquet('gs://de-zoomcamp-2026-project-bucket/weekly_housing_market_data_most_recent.parquet')

In [20]:
df_housing_gcs.printSchema()

root
 |-- PERIOD_BEGIN: timestamp (nullable = true)
 |-- PERIOD_END: timestamp (nullable = true)
 |-- REGION_TYPE: string (nullable = true)
 |-- REGION_TYPE_ID: integer (nullable = true)
 |-- REGION_NAME: string (nullable = true)
 |-- REGION_ID: integer (nullable = true)
 |-- DURATION: string (nullable = true)
 |-- ADJUSTED_AVERAGE_NEW_LISTINGS: double (nullable = true)
 |-- ADJUSTED_AVERAGE_NEW_LISTINGS_YOY: double (nullable = true)
 |-- AVERAGE_PENDING_SALES_LISTING_UPDATES: double (nullable = true)
 |-- AVERAGE_PENDING_SALES_LISTING_UPDATES_YOY: double (nullable = true)
 |-- OFF_MARKET_IN_TWO_WEEKS: integer (nullable = true)
 |-- OFF_MARKET_IN_TWO_WEEKS_YOY: double (nullable = true)
 |-- ADJUSTED_AVERAGE_HOMES_SOLD: double (nullable = true)
 |-- ADJUSTED_AVERAGE_HOMES_SOLD_YOY: double (nullable = true)
 |-- MEDIAN_NEW_LISTING_PRICE: double (nullable = true)
 |-- MEDIAN_NEW_LISTING_PRICE_YOY: double (nullable = true)
 |-- MEDIAN_SALE_PRICE: double (nullable = true)
 |-- MEDIAN_SALE_P

In [21]:
df_housing_gcs.show(truncate= False, n= 10)

+-------------------+-------------------+-----------+--------------+----------------------------+---------+--------+-----------------------------+---------------------------------+-------------------------------------+-----------------------------------------+-----------------------+---------------------------+---------------------------+-------------------------------+------------------------+----------------------------+-----------------+---------------------+--------------------+------------------------+-----------------------+---------------------------+---------------+-------------------+---------------------+-------------------------+----------------------------------------+--------------------------------------------+----------------+--------------------+---------------+-------------------+-------------------+-----------------------+--------------------------+------------------------------+----------------+--------------------+-----------------------+
|PERIOD_BEGIN       |PERIOD

In [22]:
print(df_housing_gcs.rdd.getNumPartitions)

<bound method RDD.getNumPartitions of MapPartitionsRDD[11] at javaToPython at NativeMethodAccessorImpl.java:0>


In [ ]:
bq_path = 'project-0c3c5223-416f-4242-b0f.test_dataset.market_housing_data_consolidated'

#Repartitioning the data before writing to bq
df_housing_gcs = df_housing_gcs.repartition(10)

df_housing_gcs \
    .write.format('bigquery') \
    .mode('overwrite') \
    .option('partitionField', 'PERIOD_BEGIN') \
    .option('partitionType', 'MONTH') \
    .option('clusteredFields', 'REGION_NAME') \
    .save(bq_path)

In [24]:
spark.stop()